# M7 · Observabilidad · Instrumentar un agente con Langfuse

> Coges el agente de recompensas de M1 (One Piece, ReAct con varias tools) y le pones **Langfuse** por encima: cada llamada al modelo, cada tool y cada paso de razonamiento se convierte en un **span** que inspeccionas en un dashboard.

## Qué aprendemos

1. **Vocabulario** de observabilidad LLM · trace · span · generation · session · score.
2. **`@observe()`** · decorador de Langfuse para instrumentar funciones Python: el trace es un **árbol de spans**.
3. **CallbackHandler de Langfuse** · integración nativa con LangChain/LangGraph (cero boilerplate).
4. **Sessions y users** · agrupar conversaciones multiturno por `session_id` y `user_id`.
5. **Scores programáticos** · asociar una métrica de calidad simple al trace.
6. **Dashboard** · cómo investigar coste, latencia, errores y outliers.

## Por qué importa

Una LLM app no es un microservicio cualquiera: no es determinista, cuesta dinero en cada request, tiene latencia bimodal (200 ms en caliente frente a 8 s en frío) y una conversación son 5-50 spans encadenados. Sin trazabilidad no hay producción. Es la **primera capa** del tramo final de la clase: observar (M7), proteger (M8) y optimizar coste y latencia (M9).

## Conecta con

- **M1 · Agentes con LangChain/LangGraph** · el agente de recompensas nació allí · aquí lo instrumentas.
- **clase-3 M3 · Evaluación de LLMs** y **clase-4 M4 · Evaluación de RAG** · los traces de hoy son el dataset de evaluación (LLM-as-judge, RAGas) de mañana.
- **M8 · Safety** · el logging de inputs/outputs te permite detectar prompt injection.
- **M9 · Cost y latency** · los tokens y el coste de cada trace alimentan las decisiones de optimización.

## Stack

`langfuse>=4.0` · `langchain` · `langchain-openai` · `langgraph`.

## Hardware

Cualquier máquina: el lab solo llama APIs (OpenAI + Langfuse Cloud). Tiempo: ~3 min en total con `gpt-5-mini`.

## Qué necesitas

- `OPENAI_API_KEY`.
- Cuenta en **Langfuse Cloud** (free tier): https://cloud.langfuse.com.
- En Langfuse, crea un proyecto y copia `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` y `LANGFUSE_BASE_URL` (por defecto `https://cloud.langfuse.com`; en la región US es `https://us.cloud.langfuse.com`).

> Si no quieres usar la nube, Langfuse se puede self-hostear con Docker. El SDK es el mismo: solo cambia `LANGFUSE_BASE_URL`.
>
> Sin claves de Langfuse el notebook se ejecuta igualmente (el agente funciona y `@observe()` no rompe nada), pero no se envía ningún trace: lo verás avisado en cada celda.

## Guion del laboratorio

1. **Setup**: instalar dependencias y cargar API keys.
2. **Conceptos**: trace, span, generation, session, score.
3. **Cliente Langfuse**: inicializar y comprobar la conexión.
4. **Hola mundo de la observabilidad**: una función con `@observe()`.
5. **Sessions y users**: agrupar conversaciones.
6. **El agente de recompensas de M1**: lo recuperamos en miniatura.
7. **Instrumentación nativa**: el CallbackHandler de Langfuse con LangGraph.
8. **Sesión completa**: 3 preguntas en el mismo `session_id`.
9. **Scores**: añadir una señal de calidad programática.
10. **Inspeccionar el dashboard**: qué mirar y por qué.
11. **Recap y retos**.

## 1. Setup

### 1.1. Instalar dependencias

- `langfuse`: el SDK de observabilidad (v4, basado en OpenTelemetry).
- `langchain`, `langchain-openai`, `langgraph`: el agente, igual que en M1.

> En Colab tarda unos segundos. La opción `-q` reduce el ruido.

In [1]:
%pip install -q "langfuse>=4.0,<5" langchain langchain-openai langgraph

Note: you may need to restart the kernel to use updated packages.


### 1.2. Cargar las API keys

Necesitamos `OPENAI_API_KEY` (obligatoria) y las claves de Langfuse: `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` y `LANGFUSE_BASE_URL`. El patrón es el de siempre: variable de entorno, secretos de Colab o entrada manual con `getpass`.

> Las claves de Langfuse las generas en `Settings -> API Keys` de tu proyecto en https://cloud.langfuse.com. Si pulsas Enter sin escribir nada, el notebook sigue sin enviar traces.

In [2]:
import os
from getpass import getpass


def cargar_clave(nombre: str, obligatoria: bool = True) -> bool:
    """Busca la clave en el entorno, en los secretos de Colab o la pide a mano. Devuelve True si queda cargada."""
    if os.environ.get(nombre):
        print(f"{nombre}: encontrada en las variables de entorno.")
        return True

    # Intento 2: secretos de Google Colab.
    try:
        from google.colab import userdata

        valor = userdata.get(nombre)
        if valor:
            os.environ[nombre] = valor
            print(f"{nombre}: cargada desde los secretos de Colab.")
            return True
    except Exception:
        pass

    # Intento 3: entrada manual.
    try:
        aviso = "" if obligatoria else " (Enter para saltar)"
        valor = getpass(f"Introduce tu {nombre}{aviso}: ").strip()
    except Exception:
        valor = ""  # entorno sin entrada interactiva
    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: cargada desde la entrada manual.")
        return True
    if obligatoria:
        raise RuntimeError(f"Falta {nombre}: es obligatoria para este lab.")
    print(f"{nombre}: no disponible.")
    return False


cargar_clave("OPENAI_API_KEY")
tiene_pub = cargar_clave("LANGFUSE_PUBLIC_KEY", obligatoria=False)
tiene_sec = cargar_clave("LANGFUSE_SECRET_KEY", obligatoria=False)
TIENE_CLAVES_LANGFUSE = tiene_pub and tiene_sec

# LANGFUSE_HOST es el nombre antiguo (deprecado en el SDK v4): lo aceptamos por compatibilidad.
os.environ.setdefault(
    "LANGFUSE_BASE_URL", os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")
)
print(f"LANGFUSE_BASE_URL = {os.environ['LANGFUSE_BASE_URL']}")

OPENAI_API_KEY: encontrada en las variables de entorno.
LANGFUSE_PUBLIC_KEY: encontrada en las variables de entorno.
LANGFUSE_SECRET_KEY: encontrada en las variables de entorno.
LANGFUSE_BASE_URL = http://localhost:3000


## 2. Conceptos clave de observabilidad LLM

Antes de tocar código, fijamos el vocabulario. Lo verás en cualquier herramienta (Langfuse, LangSmith, Phoenix, OpenTelemetry GenAI):

| Concepto | Qué es |
|---|---|
| **Trace** | Una unidad de trabajo de alto nivel: por ejemplo, "responder a esta pregunta del usuario" |
| **Span** | Un paso dentro del trace: una llamada a una tool, un retriever, un cálculo |
| **Generation** | Un span especial: una llamada a un LLM. Lleva modelo, tokens y coste |
| **Session** | Conjunto de traces que pertenecen a la misma conversación (`session_id`) |
| **User** | Conjunto de traces de un mismo usuario final (`user_id`) |
| **Score** | Una métrica de calidad asociada a un trace o span (`accuracy`, `helpfulness`, `thumbs`) |
| **Tag** | Etiqueta libre para filtrar (`env=prod`, `feature=rag`) |

Visualmente, un trace es un **árbol**:

```
trace: "Recompensa total de los Sombrero de Paja"
  span: invocar_agente
    generation: gpt-5-mini decide tool
    span: tool buscar_recompensa (Luffy)
    span: tool buscar_recompensa (Zoro)
    ...
    span: tool calculadora
    generation: gpt-5-mini redacta la respuesta
```

## 3. Inicializar el cliente Langfuse

El SDK v4 lee las variables de entorno automáticamente y se obtiene con `get_client()` (un singleton: todas las celdas comparten el mismo cliente).

Antes de crearlo comprobamos que las claves funcionan con una llamada directa a la API pública de Langfuse. Si no hay claves o son incorrectas, **desactivamos el envío** (`LANGFUSE_TRACING_ENABLED=false`): el resto del notebook se ejecuta igual, pero sin mandar traces. Así evitamos errores de exportación en segundo plano.

In [3]:
import base64
import json
import urllib.error
import urllib.request


def comprobar_langfuse() -> bool:
    """Llama a /api/public/projects con las claves. True si Langfuse las acepta."""
    if not TIENE_CLAVES_LANGFUSE:
        print("Sin claves de Langfuse: los traces NO se enviarán (el notebook sigue funcionando).")
        return False
    credenciales = f"{os.environ['LANGFUSE_PUBLIC_KEY']}:{os.environ['LANGFUSE_SECRET_KEY']}"
    peticion = urllib.request.Request(
        os.environ["LANGFUSE_BASE_URL"].rstrip("/") + "/api/public/projects",
        headers={"Authorization": "Basic " + base64.b64encode(credenciales.encode()).decode()},
    )
    try:
        with urllib.request.urlopen(peticion, timeout=10) as r:
            proyectos = json.loads(r.read()).get("data", [])
        print(f"Conectado a Langfuse en {os.environ['LANGFUSE_BASE_URL']} · proyecto: {proyectos[0]['name'] if proyectos else '?'}")
        return True
    except urllib.error.HTTPError as error:
        print(f"Langfuse rechaza las claves (HTTP {error.code}): revisa clave pública, secreta y región.")
    except Exception as error:
        print(f"No se pudo conectar a Langfuse ({type(error).__name__}): {error}")
    print("El notebook seguirá ejecutándose, pero los traces no se enviarán.")
    return False


LANGFUSE_OK = comprobar_langfuse()

if not LANGFUSE_OK:
    # Modo sin envío: el SDK necesita algún valor de clave para no avisar en cada llamada.
    os.environ["LANGFUSE_TRACING_ENABLED"] = "false"
    os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-sin-clave")
    os.environ.setdefault("LANGFUSE_SECRET_KEY", "sin-clave")

from langfuse import get_client

langfuse = get_client()
print("Cliente Langfuse listo ·", "enviando traces" if LANGFUSE_OK else "modo local (sin envío)")

Conectado a Langfuse en http://localhost:3000 · proyecto: pta-ia-03


Cliente Langfuse listo · enviando traces


## 4. Hola mundo de la observabilidad: `@observe()`

La forma más rápida de instrumentar código es el decorador `@observe()`. Cualquier función que decoras se convierte en un **span**. Si dentro llama a otra función observada, esa pasa a ser un **span hijo**: el árbol se construye solo.

Con `as_type="generation"` marcamos un span como llamada a modelo, y con `langfuse.update_current_generation(...)` le pasamos el modelo y los tokens: con eso Langfuse calcula el **coste** que verás en el dashboard.

Empezamos con algo trivial: dos funciones encadenadas y una llamada al LLM.

In [4]:
from langfuse import observe
from openai import OpenAI

cliente_openai = OpenAI()
MODELO = "gpt-5-mini"


@observe()
def normalizar(texto: str) -> str:
    """Limpia espacios y pasa a minúsculas: función trivial, pero queda como span."""
    return " ".join(texto.lower().split())


@observe(as_type="generation")
def saludo_llm(nombre: str) -> str:
    """Llama al LLM con un saludo personalizado. as_type='generation' marca el span como llamada a modelo."""
    respuesta = cliente_openai.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Saludas en una frase, en español y con tono entusiasta."},
            {"role": "user", "content": f"Saluda a {nombre}."},
        ],
    )
    # Modelo + tokens: Langfuse los usa para calcular el coste de esta generation.
    langfuse.update_current_generation(
        model=MODELO,
        usage_details={
            "input": respuesta.usage.prompt_tokens,
            "output": respuesta.usage.completion_tokens,
        },
    )
    return respuesta.choices[0].message.content


@observe()
def hola_mundo_observado(nombre: str) -> str:
    """Orquesta dos pasos: normalizar el nombre y pedir un saludo al LLM."""
    limpio = normalizar(nombre)
    return saludo_llm(limpio)


print(hola_mundo_observado("   Roronoa Zoro   "))

# Forzamos el envío para no esperar al flush automático (útil en notebooks).
langfuse.flush()
print("\nTrace enviado: ábrelo en el dashboard de Langfuse." if LANGFUSE_OK else "\n(Modo local: no se ha enviado ningún trace.)")

¡Hola, Roronoa Zoro, un saludo enorme, sigue demostrando esa fuerza, disciplina y valentía que te hacen único!

Trace enviado: ábrelo en el dashboard de Langfuse.


## 5. Sessions, users y metadata

En producción no basta con instrumentar funciones sueltas: queremos **agrupar** los traces de una misma conversación (session) y de un mismo usuario (user). También queremos **etiquetar** traces para filtrarlos después en el dashboard (entorno, versión del prompt, feature flag).

En el SDK v4 eso se hace con el context manager `propagate_attributes(...)`: aplica `session_id`, `user_id`, `tags` y `metadata` al span activo **y a todos sus hijos**. Se usa **dentro** de una función observada.

In [5]:
from langfuse import propagate_attributes


@observe()
def conversacion_con_metadata(nombre: str, session_id: str, user_id: str) -> str:
    """Mismo saludo, pero etiquetando el trace con sesión, usuario y tags."""
    with propagate_attributes(
        trace_name="conversacion_con_metadata",
        session_id=session_id,
        user_id=user_id,
        tags=["clase-5", "m7", "lab"],
        metadata={"feature": "saludo", "modelo": MODELO},
    ):
        return hola_mundo_observado(nombre)


respuesta = conversacion_con_metadata(
    "Nami",
    session_id="lab-m7-demo-01",
    user_id="alumno-keepcoding-001",
)

print(respuesta)
langfuse.flush()

¡Hola, Nami, qué alegría verte!


## 6. El agente de recompensas de M1, versión mini

Para que el lab tenga chicha, recuperamos en miniatura el agente ReAct de M1 con dos tools. Lo importante no es el agente: es **cómo se ve cuando lo trazamos**. Un solo `invoke` generará varios spans encadenados.

Usamos `create_agent` de LangChain v1 (sustituye a `create_react_agent` de `langgraph.prebuilt`, deprecado).

In [6]:
import ast
import operator

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Recompensas post-Wano en berries (tabla local fiable, como en el lab de M1).
_RECOMPENSAS = {
    "luffy": ("Monkey D. Luffy", "3.000.000.000"),
    "zoro": ("Roronoa Zoro", "1.101.000.000"),
    "nami": ("Nami", "366.000.000"),
    "sanji": ("Sanji", "1.032.000.000"),
    "usopp": ("Usopp", "500.000.000"),
    "chopper": ("Tony Tony Chopper", "1.000"),
    "robin": ("Nico Robin", "930.000.000"),
    "franky": ("Franky", "394.000.000"),
    "brook": ("Brook", "383.000.000"),
    "jinbe": ("Jinbe", "1.100.000.000"),
}


@tool
def buscar_recompensa(nombre: str) -> str:
    """Devuelve la recompensa (bounty) en berries de un Sombrero de Paja."""
    clave = nombre.lower().strip()
    for k, (completo, berries) in _RECOMPENSAS.items():
        if k in clave or clave in k:
            return f"{completo}: {berries} berries."
    return f"No tengo la recompensa de {nombre!r}."


_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}


def _eval_seguro(n):
    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
        return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
        return _OPS[type(n.op)](_eval_seguro(n.left), _eval_seguro(n.right))
    if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS:
        return _OPS[type(n.op)](_eval_seguro(n.operand))
    raise ValueError("Solo aritmética simple.")


@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión aritmética simple (los puntos de miles se ignoran). Úsala siempre para sumar cifras grandes."""
    try:
        return f"{_eval_seguro(ast.parse(expresion.replace('.', ''), mode='eval').body):,.0f}".replace(",", ".")
    except Exception as e:
        return f"No se pudo calcular '{expresion}': {e}"


PROMPT_SIST = (
    "Eres un experto en One Piece. Usa 'buscar_recompensa' para las cifras "
    "y SIEMPRE 'calculadora' para cualquier operación. Responde en español y de forma concisa."
)

modelo = ChatOpenAI(model=MODELO)
agente = create_agent(model=modelo, tools=[buscar_recompensa, calculadora], system_prompt=PROMPT_SIST)
print("Agente de recompensas (mini) listo.")

Agente de recompensas (mini) listo.


## 7. Instrumentación nativa con el CallbackHandler de Langfuse

Para LangChain y LangGraph no hace falta decorar nada a mano: Langfuse ofrece un **CallbackHandler** que captura cada llamada al LLM, cada tool y cada nodo del grafo. Lo pasamos vía `config={"callbacks": [handler]}`.

Así obtenemos un trace estructurado **automáticamente**, con generations tipadas (tokens y coste incluidos). La sesión, el usuario y los tags se indican con las claves especiales `langfuse_session_id`, `langfuse_user_id` y `langfuse_tags` dentro de `metadata`.

In [7]:
from langfuse.langchain import CallbackHandler

handler = CallbackHandler()


def preguntar_agente(pregunta: str, session_id: str, user_id: str) -> str:
    """Invoca el agente con el callback de Langfuse y metadata de sesión/usuario."""
    config = {
        "callbacks": [handler],
        "metadata": {
            "langfuse_session_id": session_id,
            "langfuse_user_id": user_id,
            "langfuse_tags": ["clase-5", "m7", "agente-recompensas"],
        },
    }
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config=config,
    )
    return resultado["messages"][-1].content


respuesta = preguntar_agente(
    "¿Cuál es la recompensa de Luffy?",
    session_id="lab-m7-recompensas",
    user_id="alumno-keepcoding-001",
)
print(respuesta)
langfuse.flush()

La recompensa de Monkey D. Luffy es de 3.000.000.000 berries.


## 8. Sesión completa: tres preguntas, un solo `session_id`

Una conversación real es **multiturno**. Vamos a lanzar tres preguntas con el **mismo** `session_id`. En el dashboard verás los tres traces agrupados como una sesión, con coste y latencia agregados.

> Importante: en este lab cada pregunta es independiente (el agente no recuerda nada entre llamadas). Aun así, Langfuse las agrupa en la misma sesión para analizar la conversación completa. Si quieres memoria real, usa un checkpointer (`InMemorySaver`) como en M1.

In [8]:
SESSION = "lab-m7-sesion-completa"
USER = "alumno-keepcoding-001"

preguntas = [
    "¿Cuál es la recompensa de Zoro?",
    "¿Y la de Sanji?",
    "Suma las recompensas de Luffy, Zoro y Sanji y dime el total en berries.",
]

for i, p in enumerate(preguntas, 1):
    print(f"\n--- Turno {i} ---")
    print(f"USUARIO: {p}")
    try:
        r = preguntar_agente(p, session_id=SESSION, user_id=USER)
        print(f"AGENTE: {r}")
    except Exception as e:
        print(f"Fallo en el turno {i} ({type(e).__name__}): {e}")

langfuse.flush()
print("\nLos tres traces están agrupados en la sesión", SESSION)


--- Turno 1 ---
USUARIO: ¿Cuál es la recompensa de Zoro?


AGENTE: La recompensa de Roronoa Zoro es 1.101.000.000 berries.

--- Turno 2 ---
USUARIO: ¿Y la de Sanji?


AGENTE: La recompensa de Sanji es 1.032.000.000 berries.

--- Turno 3 ---
USUARIO: Suma las recompensas de Luffy, Zoro y Sanji y dime el total en berries.


AGENTE: Total de las recompensas:
- Monkey D. Luffy: 3.000.000.000 berries
- Roronoa Zoro: 1.101.000.000 berries
- Vinsmoke Sanji: 1.032.000.000 berries

Suma total: 5.133.000.000 berries.

Los tres traces están agrupados en la sesión lab-m7-sesion-completa


## 9. Scores: añadir una señal de calidad

Un score es una métrica numérica o categórica asociada a un trace. Es la **moneda** con la que medimos calidad en LLM apps. Tres orígenes típicos:

- **Manual**: thumbs up/down en la UI del producto (`1` o `0`).
- **LLM-as-judge**: otro LLM evalúa la respuesta (lo viste en clase-3 M3 y, para RAG, con RAGas en clase-4 M4).
- **Programático**: regex, longitud, presencia de palabras clave.

Aquí hacemos uno programático simple: si la respuesta contiene un número, `contiene_numero=1`; si no, `0`.

Fíjate en la estructura: envolvemos la llamada al agente en una función con `@observe()`. El CallbackHandler cuelga sus spans **dentro** de ese trace (árbol completo: nuestra función, el grafo, el LLM y las tools) y `langfuse.score_current_trace(...)` asocia el score al trace activo.

In [9]:
import re


@observe()
def preguntar_con_score(pregunta: str, session_id: str, user_id: str) -> str:
    """Trace raíz con sesión y usuario: dentro corre el agente y al final se registra un score."""
    with propagate_attributes(
        trace_name="agente-recompensas-con-score",
        session_id=session_id,
        user_id=user_id,
        tags=["clase-5", "m7", "score"],
    ):
        resultado = agente.invoke(
            {"messages": [{"role": "user", "content": pregunta}]},
            config={"callbacks": [handler]},
        )
        respuesta = resultado["messages"][-1].content

        valor = 1 if re.search(r"\d", respuesta) else 0
        if LANGFUSE_OK:
            langfuse.score_current_trace(
                name="contiene_numero",
                value=valor,
                data_type="NUMERIC",
                comment="1 si la respuesta incluye al menos una cifra",
            )
            try:
                print("Trace en Langfuse:", langfuse.get_trace_url())
            except Exception:
                print("Trace id:", langfuse.get_current_trace_id())
        print(f"Score contiene_numero = {valor}")
        return respuesta


print(preguntar_con_score(
    "¿Cuánto suman las recompensas de Nami y Robin?",
    session_id="lab-m7-scores",
    user_id="alumno-keepcoding-001",
))
langfuse.flush()

Trace en Langfuse: http://localhost:3000/project/kc-pta-ia-03/traces/440ef3a5208c4e3926ff15465c39011b
Score contiene_numero = 1
La suma de las recompensas es 1.296.000.000 berries.


## 10. Inspeccionar el dashboard

Abre https://cloud.langfuse.com y entra en tu proyecto. Tres sitios importan:

### Traces
Lista cronológica de todos los traces. Filtra por:
- `tags = m7`
- `user_id = alumno-keepcoding-001`
- `session_id = lab-m7-sesion-completa`

Pincha en uno: verás el árbol con cada span, los inputs/outputs, los tokens, el coste estimado y la latencia desglosada por paso. Ahí detectas qué tool o qué generation se come el tiempo.

### Sessions
Agrupa traces. Útil cuando una conversación tiene 20 turnos: te muestra el flujo completo en orden, con coste y latencia acumulados.

### Dashboards
Vistas agregadas:
- Coste por día y por modelo.
- Tokens por modelo.
- Latencia P50/P95.
- Distribución de scores (por ejemplo, `contiene_numero`).

> Si los traces no aparecen, espera unos segundos: Langfuse envía en lotes. `langfuse.flush()` fuerza el envío inmediato (lo hacemos en cada celda del lab).

## 11. Recap y retos

### Qué has construido

- Un agente ReAct con dos tools, instrumentado **de extremo a extremo** con Langfuse.
- Traces con **árbol de spans** (función observada -> grafo -> LLM y tools).
- Una **sesión completa** con tres turnos agrupados por `session_id`.
- Un **score programático** asociado al trace.

### Ideas clave

- `@observe()` es la entrada más rápida: un decorador y listo.
- Para LangChain/LangGraph, el **CallbackHandler** instrumenta el grafo entero sin tocar el código del agente.
- `session_id` y `user_id` son **obligatorios** en producción: sin ellos no puedes investigar nada.
- Los **scores** son la base de la evaluación automática (LLM-as-judge de clase-3, RAGas de clase-4) aplicada sobre tráfico real.

### Retos para practicar

1. **Modelo más barato como fallback**: añade una versión del agente que use `gpt-5-nano` si el coste del trace supera un umbral, y registra en un score el modelo usado.
2. **PII masking**: muchos usuarios pegan emails o teléfonos. Usa el parámetro `mask` de `Langfuse(...)` para enmascarar PII antes de enviar inputs y outputs.
3. **Replay**: descarga un trace fallido con la API de Langfuse y conviértelo en un test de regresión local.
4. **Self-host**: levanta Langfuse en local con Docker (`docker compose up`) y envía los traces a `http://localhost:3000` cambiando `LANGFUSE_BASE_URL`.

> **Siguiente: M8 · Safety.** Ya ves todo lo que entra y sale del agente; ahora toca protegerlo: guardrails, OWASP para LLMs y red teaming contra prompt injection. Los traces de hoy son justo donde detectarás esos ataques.